In [26]:
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal
from typing import List

In [27]:
@dataclass
class Operation:
    name: str
    amount: Decimal
    date: datetime
    balance_after: Decimal
    status: str

In [28]:
class Account:
    def __init__(self, account_holder: str, balance=0):
        balance = self._to_decimal(balance)
        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self.holder = account_holder
        self._balance = Account._to_decimal(balance)
        self.operations_history: List[Operation] = []

    @staticmethod
    def _to_decimal(value) -> Decimal:
        # лучше через str
        return Decimal(str(value))

    def deposit(self, amount) -> bool:
        amount = self._to_decimal(amount)
        if amount <= 0:
            self._log("deposit", amount, "fail")
            return False

        self._balance += amount # normalized by Decimal(str())
        self._log("deposit", amount, "success")
        return True

    def withdraw(self, amount) -> bool:
        """Неудачная попытка сохраняется в истории"""
        amount = self._to_decimal(amount)
        if amount <= 0 or amount > self._balance:
            self._log("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._log("withdraw", amount, "success")
        return True

    def get_balance(self) -> Decimal:
        return self._balance

    def get_history(self) -> List[dict]:
        """Возвращает историю операций"""
        return [self._operation_to_dict(op) for op in self.operations_history]

    def _operation_to_dict(self, op: Operation) -> dict:
        return {
            "operation": op.name,
            "amount": op.amount,
            "date": op.date.strftime("%d-%m-%Y %H:%M:%S"),
            "balance_after": op.balance_after,
            "status": op.status,
        }

    def _log(self, name: str, amount: Decimal, status: str):
        self.operations_history.append(
            Operation(name, amount, datetime.now(), self._balance, status)
        )

In [29]:
acc = Account("Ivan", 0)
print(acc.holder)
acc.get_balance()

Ivan


Decimal('0')

In [30]:
acc.deposit(50)
acc.get_balance()

Decimal('50')

In [31]:
success = acc.withdraw(500)
success

False

In [32]:
acc.get_history()[-1]  # последняя операция

{'operation': 'withdraw',
 'amount': Decimal('500'),
 'date': '08-11-2026 00:52:19',
 'balance_after': Decimal('50'),
 'status': 'fail'}

In [33]:
success = acc.withdraw(30)
success

True

In [34]:
acc.get_history()[-1]  # последняя операция

{'operation': 'withdraw',
 'amount': Decimal('30'),
 'date': '08-11-2026 00:52:27',
 'balance_after': Decimal('20'),
 'status': 'success'}

In [35]:
acc.withdraw(-10)  # отрицательная сумма не проходит

False

In [36]:
acc.get_balance()

Decimal('20')

In [37]:
for op in acc.get_history():
    print(op["operation"], op["status"])

deposit success
withdraw fail
withdraw success
withdraw fail
